# MSCAF-TransUNet + Reverse Attention s0 r8

Mục tiêu:
- Train MSCAF-TransUNet với Reverse Attention module trên decoder skip connections.
- Giữ RA ở skip index 0 như run vừa rồi nhưng tăng reduction lên 8 để làm RA nhẹ hơn, kỳ vọng giảm HD95.
- Train 150 epochs, evaluate, export metrics và artifacts về Google Drive.

Cấu hình:
- `ATTENTION_MODE = "pre_hidden"` encoder-side MSCAF
- `ATTENTION_SCALES = "1/16"`
- `ATTENTION_REDUCTION = 16`
- `RA_MODE = "ra_skip"`
- `RA_SCALES = "0"`
- `RA_REDUCTION = 8`

Kết quả mong đợi:
- Snapshot suffix: `_attn-pre_hidden-1_16-r16_ra-ra_skip-s0-r8`
- So sánh chính với run trước: `pre_hidden 1/16` không RA và run `ra_skip-s0-r4` đã chạy.
- Mục tiêu tốt: Pancreas DSC cao hơn 55.39% và Pancreas HD95 thấp hơn 19.29.

Lưu ý chạy song song:
- Mỗi notebook nên mở ở một Colab runtime/session riêng.
- Snapshot name khác nhau nên artifact export về Drive không ghi đè nhau.


In [1]:
from pathlib import Path

# Storage / persistence
USE_GOOGLE_DRIVE = True
WORKSPACE_ROOT = Path('/content')
FORCE_REBUILD_PROJECT = True
EXPORT_TO_DRIVE = True
DRIVE_EXPORT_DIR = Path('/content/drive/MyDrive/transunet_colab_outputs')
PERSIST_CHECKPOINTS_TO_DRIVE = True

# Code bootstrap
REPO_SOURCE = 'embedded'
PROJECT_DIRNAME = 'TransUNet-Medical-Image-Segmentation'
DRIVE_REPO_DIR = Path('/content/drive/MyDrive/TransUNet-Medical-Image-Segmentation')
DRIVE_REPO_ZIP = Path('/content/drive/MyDrive/TransUNet-Medical-Image-Segmentation.zip')

# Dataset and pretrained weights
DRIVE_SEARCH_ROOT = Path('/content/drive/MyDrive')
AUTO_DISCOVER_DRIVE_DATASET = True
AUTO_DISCOVER_DRIVE_WEIGHT = True
FALLBACK_DATA_SOURCE_TO_DOWNLOAD = True
FALLBACK_WEIGHT_SOURCE_TO_DOWNLOAD = True

DATA_SOURCE = 'drive'
DRIVE_DATASET_DIR = Path('/content/drive/MyDrive/datasets/Synapse')
COPY_DATA_TO_RUNTIME = False
SYNAPSE_ARCHIVE_FILE_ID = '1BvpY0g9mKkkhdHpAX1HqDw8iTJNbFuwq'
SYNAPSE_ARCHIVE_NAME = 'project_TransUNet.zip'

WEIGHTS_SOURCE = 'drive'
DRIVE_WEIGHT_FILE = Path('/content/drive/MyDrive/transunet/R50+ViT-B_16.npz')
WEIGHT_DOWNLOAD_URLS = [
    'https://huggingface.co/kenton-li/nnSAM/resolve/main/R50%2BViT-B_16.npz?download=true',
    'https://storage.googleapis.com/vit_models/imagenet21k/R50+ViT-B_16.npz',
    'https://storage.googleapis.com/vit_models/imagenet21k/R50-ViT-B_16.npz',
]

# Experiment
RUN_PROFILE = 'full'
ATTENTION_MODE = 'pre_hidden'
ATTENTION_SCALES = '1/16'
ATTENTION_REDUCTION = 16

# Reverse Attention (decoder-side)
RA_MODE = 'ra_skip'
RA_SCALES = '0'
RA_REDUCTION = 8

RUN_TRAIN = True
RUN_TEST = True
SAVE_NIFTI = True
ZIP_ARTIFACTS = True
FORCE_REINSTALL_PACKAGES = True

OVERRIDES = {
    'dataset': 'Synapse',
    'img_size': 224,
    'vit_name': 'R50-ViT-B_16',
    'vit_patches_size': 16,
    'n_skip': 3,
    'num_classes': 9,
    'seed': 1234,
    'deterministic': 1,
    'max_iterations': 30000,
    'num_workers': 2,
    'max_train_samples': 0,
    'max_epochs': 150,
    'batch_size': None,
    'base_lr': None,
}


In [2]:

import base64
import io
import os
import shutil
import sys
import zipfile

def resolve_colab_environment():
    try:
        import google.colab  # noqa: F401
        return True, None
    except ImportError as exc:
        return False, exc

IN_COLAB, COLAB_IMPORT_ERROR = resolve_colab_environment()

if USE_GOOGLE_DRIVE:
    if IN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    elif Path("/content/drive/MyDrive").exists():
        print("google.colab import failed, but /content/drive is already present. Reusing existing mount.")
    else:
        raise RuntimeError(
            "USE_GOOGLE_DRIVE=True nhưng kernel hiện tại chưa mount được Google Drive. "
            "Nếu đây là Colab kernel trong VS Code, hãy chạy lại cell này và hoàn tất bước xác thực Drive. "
            f"Import error gốc: {COLAB_IMPORT_ERROR}"
        )

PROJECT_DIR = WORKSPACE_ROOT / PROJECT_DIRNAME

def reset_path(path):
    if path.is_symlink() or path.is_file():
        path.unlink()
    elif path.exists():
        shutil.rmtree(path)

def materialize_from_embedded(project_dir):
    snapshot_b64 = (
        '...'
        'BgAAAAAOAA4AmwMAAAtiAAAAAA=='
    )
    payload = base64.b64decode(snapshot_b64)
    project_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        archive.extractall(project_dir)

if FORCE_REBUILD_PROJECT and PROJECT_DIR.exists():
    reset_path(PROJECT_DIR)

if REPO_SOURCE == "embedded":
    materialize_from_embedded(PROJECT_DIR)
elif REPO_SOURCE == "drive_repo":
    if not DRIVE_REPO_DIR.exists():
        raise FileNotFoundError(f"Drive repo not found: {DRIVE_REPO_DIR}")
    shutil.copytree(DRIVE_REPO_DIR, PROJECT_DIR)
elif REPO_SOURCE == "drive_zip":
    if not DRIVE_REPO_ZIP.exists():
        raise FileNotFoundError(f"Drive repo zip not found: {DRIVE_REPO_ZIP}")
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DRIVE_REPO_ZIP) as archive:
        archive.extractall(PROJECT_DIR)
else:
    raise ValueError(f"Unsupported REPO_SOURCE: {REPO_SOURCE}")

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# Force local project packages to win over similarly named third-party packages on Colab.
for package_dir in ("datasets", "networks"):
    init_file = PROJECT_DIR / package_dir / "__init__.py"
    init_file.parent.mkdir(parents=True, exist_ok=True)
    if not init_file.exists():
        init_file.write_text('"""Project package."""\n', encoding="utf-8")

TRAINER_PATCH = r'''import argparse
import logging
import os
import random
import sys
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from tensorboardX import SummaryWriter
from torch.nn.modules.loss import CrossEntropyLoss
from torch.utils.data import DataLoader
from tqdm import tqdm
from utils import DiceLoss
from torchvision import transforms

def _format_duration(seconds):
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    if h > 0:
        return f"{h}h {m:02d}m {s:02d}s"
    return f"{m}m {s:02d}s"


def trainer_synapse(args, model, snapshot_path):
    from datasets.synapse import Synapse_dataset, RandomGenerator
    logging.basicConfig(filename=snapshot_path + "/log.txt", level=logging.INFO,
                        format='[%(asctime)s.%(msecs)03d] %(message)s', datefmt='%H:%M:%S')
    logging.getLogger().addHandler(logging.StreamHandler(sys.stdout))
    logging.info(str(args))
    base_lr = args.base_lr
    num_classes = args.num_classes
    batch_size = args.batch_size * args.n_gpu
    device = next(model.parameters()).device
    checkpoint_dir = os.environ.get('TRANSUNET_CHECKPOINT_DIR', snapshot_path)
    mid_epoch_checkpoint_interval = int(os.environ.get('TRANSUNET_MID_EPOCH_SAVE_ITERS', '200'))
    iter_log_interval = int(os.environ.get('TRANSUNET_ITER_LOG_INTERVAL', '10'))
    db_train = Synapse_dataset(base_dir=args.root_path, list_dir=args.list_dir, split="train",
                               max_samples=args.max_train_samples,
                               transform=transforms.Compose(
                                   [RandomGenerator(output_size=[args.img_size, args.img_size])]))
    print("The length of train set is: {}".format(len(db_train)))

    def worker_init_fn(worker_id):
        random.seed(args.seed + worker_id)

    trainloader = DataLoader(db_train, batch_size=batch_size, shuffle=True, num_workers=args.num_workers, pin_memory=(device.type == 'cuda'),
                             worker_init_fn=worker_init_fn)
    total_batches_per_epoch = len(trainloader)
    if args.n_gpu > 1 and device.type == 'cuda':
        model = nn.DataParallel(model)
    model.train()
    ce_loss = CrossEntropyLoss()
    dice_loss = DiceLoss(num_classes)
    optimizer = optim.SGD(model.parameters(), lr=base_lr, momentum=0.9, weight_decay=0.0001)
    writer = SummaryWriter(snapshot_path + '/log')
    iter_num = 0
    start_epoch = 0
    start_batch = 0
    checkpoint_file = os.path.join(checkpoint_dir, 'latest_checkpoint.pth')
    if os.path.exists(checkpoint_file):
        checkpoint = torch.load(checkpoint_file, weights_only=False)
        load_result = model.load_state_dict(checkpoint['model_state'], strict=False)
        if load_result.missing_keys:
            logging.info(f"Missing {len(load_result.missing_keys)} keys when loading checkpoint (likely new modules, will use default init)")
            for k in load_result.missing_keys[:5]:
                logging.info(f"  - {k}")
        if load_result.unexpected_keys:
            logging.info(f"Unexpected {len(load_result.unexpected_keys)} keys in checkpoint (will be ignored)")
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        start_epoch = checkpoint['epoch'] + 1
        iter_num = checkpoint['iter_num']
        start_batch = checkpoint.get('batch_idx', 0)
        if start_batch > 0:
            logging.info(f"Resumed from checkpoint epoch {checkpoint['epoch']}, batch {start_batch}, iter {iter_num} (mid-epoch)")
        else:
            logging.info(f"Resumed from checkpoint epoch {checkpoint['epoch']}, iter {iter_num}")
            start_batch = 0
    max_epoch = args.max_epochs
    max_iterations = args.max_epochs * total_batches_per_epoch
    logging.info("{} iterations per epoch. {} max iterations ".format(total_batches_per_epoch, max_iterations))
    if start_epoch > 0:
        logging.info(f"Resuming from epoch {start_epoch}/{max_epoch} (skipping {start_epoch} completed epochs)")
    best_performance = 0.0
    training_start_time = time.time()
    lr_ = base_lr
    for epoch_num in range(start_epoch, max_epoch):
        epoch_start_time = time.time()
        epoch_loss_sum = 0.0
        epoch_ce_sum = 0.0
        epoch_batches = 0
        model.train()

        print(f"\n{'='*70}", flush=True)
        print(f"  Epoch {epoch_num + 1}/{max_epoch}  |  Batches: {total_batches_per_epoch}  |  LR: {lr_:.6f}", flush=True)
        print(f"{'='*70}", flush=True)

        skip_batches = start_batch if epoch_num == start_epoch and start_batch > 0 else 0
        if skip_batches > 0:
            print(f"  Skipping {skip_batches} already-completed batches from mid-epoch checkpoint...", flush=True)

        for i_batch, sampled_batch in enumerate(trainloader):
            if i_batch < skip_batches:
                continue

            image_batch, label_batch = sampled_batch['image'], sampled_batch['label']
            image_batch = image_batch.to(device)
            label_batch = label_batch.to(device)
            outputs = model(image_batch)
            loss_ce = ce_loss(outputs, label_batch[:].long())
            loss_dice = dice_loss(outputs, label_batch, softmax=True)
            loss = 0.5 * loss_ce + 0.5 * loss_dice
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            lr_ = base_lr * (1.0 - iter_num / max_iterations) ** 0.9
            for param_group in optimizer.param_groups:
                param_group['lr'] = lr_

            iter_num = iter_num + 1
            epoch_loss_sum += loss.item()
            epoch_ce_sum += loss_ce.item()
            epoch_batches += 1
            writer.add_scalar('info/lr', lr_, iter_num)
            writer.add_scalar('info/total_loss', loss, iter_num)
            writer.add_scalar('info/loss_ce', loss_ce, iter_num)

            if epoch_batches % iter_log_interval == 0:
                batch_elapsed = time.time() - epoch_start_time
                batch_speed = batch_elapsed / epoch_batches
                remaining_batches = total_batches_per_epoch - i_batch - 1
                batch_eta = remaining_batches * batch_speed
                print(
                    f"  [{i_batch + 1}/{total_batches_per_epoch}] "
                    f"loss={loss.item():.4f} ce={loss_ce.item():.4f} dice={loss_dice.item():.4f} "
                    f"lr={lr_:.6f} | "
                    f"{_format_duration(batch_elapsed)}<{_format_duration(batch_eta)}",
                    flush=True,
                )

            if iter_num % 20 == 0:
                vis_index = 1 if image_batch.size(0) > 1 else 0
                image = image_batch[vis_index, 0:1, :, :]
                image = (image - image.min()) / (image.max() - image.min())
                writer.add_image('train/Image', image, iter_num)
                outputs = torch.argmax(torch.softmax(outputs, dim=1), dim=1, keepdim=True)
                writer.add_image('train/Prediction', outputs[vis_index, ...] * 50, iter_num)
                labs = label_batch[vis_index, ...].unsqueeze(0) * 50
                writer.add_image('train/GroundTruth', labs, iter_num)

            if mid_epoch_checkpoint_interval > 0 and epoch_batches % mid_epoch_checkpoint_interval == 0:
                torch.save({
                    'epoch': epoch_num,
                    'batch_idx': i_batch + 1,
                    'iter_num': iter_num,
                    'model_state': model.state_dict(),
                    'optimizer_state': optimizer.state_dict(),
                }, os.path.join(checkpoint_dir, 'latest_checkpoint.pth'))

        epoch_elapsed = time.time() - epoch_start_time
        total_elapsed = time.time() - training_start_time
        completed_epochs = epoch_num - start_epoch + 1
        remaining_epochs = max_epoch - epoch_num - 1
        avg_epoch_time = total_elapsed / completed_epochs
        eta_seconds = remaining_epochs * avg_epoch_time
        avg_loss = epoch_loss_sum / max(epoch_batches, 1)
        avg_ce = epoch_ce_sum / max(epoch_batches, 1)
        epoch_summary = (
            f"\n>>> [Epoch {epoch_num + 1}/{max_epoch} DONE] "
            f"avg_loss={avg_loss:.4f} avg_ce={avg_ce:.4f} lr={lr_:.6f} | "
            f"epoch: {_format_duration(epoch_elapsed)} "
            f"total: {_format_duration(total_elapsed)} "
            f"ETA: {_format_duration(eta_seconds)} "
            f"({completed_epochs}/{max_epoch - start_epoch} epochs done)"
        )
        print(epoch_summary, flush=True)
        logging.info(epoch_summary)

        torch.save({
            'epoch': epoch_num,
            'batch_idx': 0,
            'iter_num': iter_num,
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
        }, os.path.join(checkpoint_dir, 'latest_checkpoint.pth'))
        save_interval = 50
        if epoch_num > int(max_epoch / 2) and (epoch_num + 1) % save_interval == 0:
            save_mode_path = os.path.join(snapshot_path, 'epoch_' + str(epoch_num) + '.pth')
            torch.save(model.state_dict(), save_mode_path)
            logging.info("save model to {}".format(save_mode_path))

        if epoch_num >= max_epoch - 1:
            save_mode_path = os.path.join(snapshot_path, 'epoch_' + str(epoch_num) + '.pth')
            torch.save(model.state_dict(), save_mode_path)
            logging.info("save model to {}".format(save_mode_path))
            break

    total_training_time = time.time() - training_start_time
    logging.info(f"Training completed in {_format_duration(total_training_time)}")
    writer.close()
    return "Training Finished!"
'''

(PROJECT_DIR / "trainer.py").write_text(TRAINER_PATCH, encoding="utf-8")
print("Patched trainer.py with real-time progress logging + mid-epoch checkpoints")

print(f"Project ready at: {PROJECT_DIR}")
for rel_path in [
    "train.py",
    "test.py",
    "trainer.py",
    "datasets/synapse.py",
    "networks/vit_seg_modeling.py",
]:
    print(" -", rel_path, "OK" if (PROJECT_DIR / rel_path).exists() else "MISSING")


Mounted at /content/drive
Patched trainer.py with real-time progress logging + mid-epoch checkpoints
Project ready at: /content/TransUNet-Medical-Image-Segmentation
 - train.py OK
 - test.py OK
 - trainer.py OK
 - datasets/synapse.py OK
 - networks/vit_seg_modeling.py OK


In [3]:

import shlex
import subprocess

def run_install(cmd):
    print("$", " ".join(shlex.quote(str(part)) for part in cmd))
    subprocess.run([str(part) for part in cmd], cwd=PROJECT_DIR, check=True)

pip_install_args = ["--upgrade"]
if FORCE_REINSTALL_PACKAGES:
    pip_install_args.append("--force-reinstall")

run_install([sys.executable, "-m", "pip", "install", *pip_install_args, "pip", "setuptools", "wheel"])

runtime_specs = [
    "numpy>=1.26,<2",
    "scipy",
    "h5py",
    "tensorboard",
    "tensorboardX",
    "ml-collections",
    "medpy",
    "SimpleITK",
    "gdown",
    "",
]

print("Installing runtime-compatible packages for Python", sys.version.split()[0])
run_install([sys.executable, "-m", "pip", "install", *pip_install_args] + runtime_specs)

import numpy as np
import torch

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name} ({gpu.total_memory / 2**30:.1f} GB)")
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("GPU not detected. Switch the Colab runtime to GPU before full training.")

$ /usr/bin/python3 -m pip install --upgrade --force-reinstall pip setuptools wheel
Installing runtime-compatible packages for Python 3.12.13
$ /usr/bin/python3 -m pip install --upgrade --force-reinstall 'numpy>=1.26,<2' scipy h5py tensorboard tensorboardX ml-collections medpy SimpleITK gdown ''
Python: 3.12.13
NumPy: 2.0.2
Torch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4 (14.6 GB)


In [4]:

import json
import shutil
import tempfile
import urllib.request
import zipfile

import gdown

def ensure_link_or_copy(source, target, copy_to_runtime=False):
    source = Path(source)
    target = Path(target)
    if not source.exists():
        raise FileNotFoundError(source)

    if target.is_symlink() or target.is_file():
        target.unlink()
    elif target.exists():
        shutil.rmtree(target)

    target.parent.mkdir(parents=True, exist_ok=True)
    if copy_to_runtime:
        if source.is_dir():
            shutil.copytree(source, target)
        else:
            shutil.copy2(source, target)
    else:
        target.symlink_to(source, target_is_directory=source.is_dir())

def is_synapse_root(candidate):
    candidate = Path(candidate)
    return (candidate / "train_npz").exists() and (candidate / "test_vol_h5").exists()

def discover_synapse_roots(search_root, max_depth=6, limit=10):
    search_root = Path(search_root)
    if not search_root.exists():
        return []

    matches = []
    seen = set()
    for train_dir in search_root.rglob("train_npz"):
        try:
            relative = train_dir.relative_to(search_root)
        except ValueError:
            continue
        if len(relative.parts) > max_depth:
            continue

        root = train_dir.parent
        if is_synapse_root(root):
            key = str(root.resolve())
            if key not in seen:
                matches.append(root)
                seen.add(key)
        if len(matches) >= limit:
            break

    return sorted(matches, key=lambda item: (len(item.parts), str(item)))

def resolve_synapse_root(candidate):
    candidate = Path(candidate)
    explicit_candidates = [
        candidate,
        candidate / "Synapse",
        DRIVE_SEARCH_ROOT / "datasets" / "Synapse",
        DRIVE_SEARCH_ROOT / "Synapse",
        DRIVE_SEARCH_ROOT / "data" / "Synapse",
    ]

    for option in explicit_candidates:
        if is_synapse_root(option):
            print("Using Synapse dataset:", option)
            return option

    if DATA_SOURCE == "drive" and AUTO_DISCOVER_DRIVE_DATASET:
        discovered = discover_synapse_roots(DRIVE_SEARCH_ROOT)
        if discovered:
            print("Auto-discovered Synapse dataset:", discovered[0])
            if len(discovered) > 1:
                print("Other candidates:")
                for extra in discovered[1:]:
                    print(" -", extra)
            return discovered[0]

    raise FileNotFoundError(
        "Không tìm thấy Synapse dataset trên Google Drive. "
        "Hãy kiểm tra DRIVE_DATASET_DIR hoặc đặt dataset sao cho có cấu trúc train_npz/ và test_vol_h5/. "
        f"Đường dẫn đã thử đầu tiên: {candidate}"
    )

def normalize_weight_files(weights_dir):
    plus_name = weights_dir / "R50+ViT-B_16.npz"
    minus_name = weights_dir / "R50-ViT-B_16.npz"

    if plus_name.exists() and not minus_name.exists():
        shutil.copy2(plus_name, minus_name)
    if minus_name.exists() and not plus_name.exists():
        shutil.copy2(minus_name, plus_name)

    if not plus_name.exists() or not minus_name.exists():
        raise FileNotFoundError(
            f"Expected both weight aliases to exist in {weights_dir}, but found plus={plus_name.exists()} minus={minus_name.exists()}"
        )
    return plus_name, minus_name

def discover_weight_files(search_root, limit=10):
    search_root = Path(search_root)
    if not search_root.exists():
        return []

    preferred = [
        search_root / "transunet" / "R50+ViT-B_16.npz",
        search_root / "transunet" / "R50-ViT-B_16.npz",
        search_root / "R50+ViT-B_16.npz",
        search_root / "R50-ViT-B_16.npz",
    ]

    matches = []
    seen = set()
    for candidate in preferred:
        if candidate.exists() and candidate.is_file():
            key = str(candidate.resolve())
            if key not in seen:
                matches.append(candidate)
                seen.add(key)

    for pattern in ("R50+ViT-B_16.npz", "R50-ViT-B_16.npz"):
        for candidate in search_root.rglob(pattern):
            if candidate.is_file():
                key = str(candidate.resolve())
                if key not in seen:
                    matches.append(candidate)
                    seen.add(key)
            if len(matches) >= limit:
                break
        if len(matches) >= limit:
            break

    return sorted(matches, key=lambda item: (len(item.parts), str(item)))

def resolve_weight_file(candidate):
    candidate = Path(candidate)
    direct_candidates = [
        candidate,
        candidate / "R50+ViT-B_16.npz",
        candidate / "R50-ViT-B_16.npz",
    ]

    for option in direct_candidates:
        if option.exists() and option.is_file():
            print("Using pretrained weight:", option)
            return option

    if WEIGHTS_SOURCE == "drive" and AUTO_DISCOVER_DRIVE_WEIGHT:
        discovered = discover_weight_files(DRIVE_SEARCH_ROOT)
        if discovered:
            print("Auto-discovered pretrained weight:", discovered[0])
            if len(discovered) > 1:
                print("Other weight candidates:")
                for extra in discovered[1:]:
                    print(" -", extra)
            return discovered[0]

    raise FileNotFoundError(
        "Không tìm thấy pretrained weight trên Google Drive. "
        "Hãy kiểm tra DRIVE_WEIGHT_FILE hoặc đặt file R50+ViT-B_16.npz vào MyDrive. "
        f"Đường dẫn đã thử đầu tiên: {candidate}"
    )

def try_download_weight(target_file, urls):
    target_file = Path(target_file)
    attempted = []
    for url in urls:
        try:
            print("Trying weight URL:", url)
            urllib.request.urlretrieve(url, target_file)
            size_mb = target_file.stat().st_size / (1024 ** 2)
            print(f"Downloaded {target_file.name}: {size_mb:.1f} MB")
            if size_mb < 100:
                raise RuntimeError(f"Downloaded file is unexpectedly small: {size_mb:.1f} MB")
            return url
        except Exception as exc:
            attempted.append({"url": url, "error": str(exc)})
            print("  Failed:", exc)
            if target_file.exists():
                target_file.unlink()
    raise RuntimeError(
        "Không tải được pretrained weight từ các URL mặc định. "
        "Hãy chuyển WEIGHTS_SOURCE='drive' và đặt file R50+ViT-B_16.npz trên Google Drive. "
        f"Chi tiết thử tải: {json.dumps(attempted, indent=2)}"
    )

def ensure_expected_synapse_layout(expected_root, discovered_root):
    expected_root = Path(expected_root)
    discovered_root = Path(discovered_root)

    if expected_root.resolve() == discovered_root.resolve():
        return expected_root

    expected_root.mkdir(parents=True, exist_ok=True)
    for folder_name in ("train_npz", "test_vol_h5"):
        source = discovered_root / folder_name
        target = expected_root / folder_name
        if not source.exists():
            raise FileNotFoundError(source)
        if target.exists() or target.is_symlink():
            if target.is_symlink() or target.is_file():
                target.unlink()
            elif target.resolve() != source.resolve():
                shutil.rmtree(target)
            else:
                continue
        target.symlink_to(source, target_is_directory=True)

    return expected_root

def download_synapse_archive(target_root):
    archive_path = target_root.parent / SYNAPSE_ARCHIVE_NAME
    extract_root = Path(tempfile.gettempdir()) / "transunet_synapse_extract_notebook"

    print("Downloading Synapse archive to:", archive_path)
    archive_path.parent.mkdir(parents=True, exist_ok=True)
    gdown.download(id=SYNAPSE_ARCHIVE_FILE_ID, output=str(archive_path), quiet=False, resume=True)

    if extract_root.exists():
        shutil.rmtree(extract_root)
    extract_root.mkdir(parents=True, exist_ok=True)

    print("Extracting archive to:", extract_root)
    with zipfile.ZipFile(archive_path, "r") as zip_file:
        zip_file.extractall(extract_root)

    candidates = discover_synapse_roots(extract_root, max_depth=12, limit=50)
    if not candidates:
        raise RuntimeError(
            f"Archive extracted under {extract_root}, but no Synapse layout was found."
        )

    chosen = candidates[0]
    print("Normalizing downloaded dataset layout from:", chosen)
    if target_root.exists():
        shutil.rmtree(target_root)
    target_root.mkdir(parents=True, exist_ok=True)

    for folder_name in ("train_npz", "test_vol_h5"):
        shutil.move(str(chosen / folder_name), str(target_root / folder_name))

    shutil.rmtree(extract_root, ignore_errors=True)
    return target_root

data_root = PROJECT_DIR / "data" / "Synapse"
train_npz_dir = data_root / "train_npz"
test_vol_dir = data_root / "test_vol_h5"
resolved_drive_root = None
source_weight = None

if DATA_SOURCE == "download":
    if not train_npz_dir.exists() or not test_vol_dir.exists():
        download_synapse_archive(data_root)
elif DATA_SOURCE == "drive":
    if is_synapse_root(data_root):
        print("Reusing existing runtime dataset:", data_root)
    else:
        try:
            resolved_drive_root = resolve_synapse_root(DRIVE_DATASET_DIR)
            ensure_link_or_copy(resolved_drive_root, data_root, copy_to_runtime=COPY_DATA_TO_RUNTIME)
        except FileNotFoundError as exc:
            if not FALLBACK_DATA_SOURCE_TO_DOWNLOAD:
                raise
            print("Drive dataset not found. Falling back to direct archive download.")
            print(exc)
            download_synapse_archive(data_root)
elif DATA_SOURCE == "existing":
    pass
else:
    raise ValueError(f"Unsupported DATA_SOURCE: {DATA_SOURCE}")

if not is_synapse_root(data_root):
    local_candidates = discover_synapse_roots(PROJECT_DIR / "data", max_depth=10, limit=20)
    if local_candidates:
        print("Normalizing downloaded dataset layout from:", local_candidates[0])
        ensure_expected_synapse_layout(data_root, local_candidates[0])

train_npz_dir = data_root / "train_npz"
test_vol_dir = data_root / "test_vol_h5"

weights_dir = PROJECT_DIR / "model" / "vit_checkpoint" / "imagenet21k"
weights_dir.mkdir(parents=True, exist_ok=True)

if WEIGHTS_SOURCE == "download":
    if not any(weights_dir.glob("R50*ViT-B_16.npz")):
        downloaded_to = weights_dir / "R50+ViT-B_16.npz"
        used_url = try_download_weight(downloaded_to, WEIGHT_DOWNLOAD_URLS)
        print("Weight source URL selected:", used_url)
elif WEIGHTS_SOURCE == "drive":
    try:
        source_weight = resolve_weight_file(DRIVE_WEIGHT_FILE)
        shutil.copy2(source_weight, weights_dir / source_weight.name)
    except FileNotFoundError as exc:
        if not FALLBACK_WEIGHT_SOURCE_TO_DOWNLOAD:
            raise
        print("Drive pretrained weight not found. Falling back to download mode.")
        print(exc)
        downloaded_to = weights_dir / "R50+ViT-B_16.npz"
        used_url = try_download_weight(downloaded_to, WEIGHT_DOWNLOAD_URLS)
        print("Weight source URL selected:", used_url)
else:
    raise ValueError(f"Unsupported WEIGHTS_SOURCE: {WEIGHTS_SOURCE}")

plus_weight, minus_weight = normalize_weight_files(weights_dir)

train_count = len(list(train_npz_dir.glob("*.npz"))) if train_npz_dir.exists() else 0
test_count = len(list(test_vol_dir.glob("*.npy.h5"))) if test_vol_dir.exists() else 0

data_summary = {
    "drive_enabled": USE_GOOGLE_DRIVE,
    "in_colab": IN_COLAB,
    "drive_mount_exists": Path("/content/drive/MyDrive").exists(),
    "data_source": DATA_SOURCE,
    "weights_source": WEIGHTS_SOURCE,
    "drive_search_root": str(DRIVE_SEARCH_ROOT),
    "fallback_data_to_download": FALLBACK_DATA_SOURCE_TO_DOWNLOAD,
    "fallback_weight_to_download": FALLBACK_WEIGHT_SOURCE_TO_DOWNLOAD,
    "resolved_drive_dataset": str(resolved_drive_root) if DATA_SOURCE == "drive" else None,
    "resolved_drive_weight": str(source_weight) if source_weight is not None else None,
    "train_npz_dir": str(train_npz_dir),
    "test_vol_dir": str(test_vol_dir),
    "train_npz_count": train_count,
    "test_volume_count": test_count,
    "weights_plus_name": str(plus_weight),
    "weights_minus_name": str(minus_weight),
}
print(json.dumps(data_summary, indent=2))

if train_count == 0 or test_count == 0:
    raise RuntimeError(
        "Synapse data is missing. If Google Drive download hits quota, switch DATA_SOURCE='drive' and point DRIVE_DATASET_DIR to a valid Synapse folder on Drive."
    )

Using Synapse dataset: /content/drive/MyDrive/datasets/Synapse
Using pretrained weight: /content/drive/MyDrive/transunet/R50+ViT-B_16.npz
{
  "drive_enabled": true,
  "in_colab": true,
  "drive_mount_exists": true,
  "data_source": "drive",
  "weights_source": "drive",
  "drive_search_root": "/content/drive/MyDrive",
  "fallback_data_to_download": true,
  "fallback_weight_to_download": true,
  "resolved_drive_dataset": "/content/drive/MyDrive/datasets/Synapse",
  "resolved_drive_weight": "/content/drive/MyDrive/transunet/R50+ViT-B_16.npz",
  "train_npz_dir": "/content/TransUNet-Medical-Image-Segmentation/data/Synapse/train_npz",
  "test_vol_dir": "/content/TransUNet-Medical-Image-Segmentation/data/Synapse/test_vol_h5",
  "train_npz_count": 2211,
  "test_volume_count": 12,
  "weights_plus_name": "/content/TransUNet-Medical-Image-Segmentation/model/vit_checkpoint/imagenet21k/R50+ViT-B_16.npz",
  "weights_minus_name": "/content/TransUNet-Medical-Image-Segmentation/model/vit_checkpoint/ima

In [5]:

# Optional: copy the prepared Synapse dataset + pretrained weight into MyDrive for later runs.
PUSH_RUNTIME_CACHE_TO_DRIVE = False
OVERWRITE_DRIVE_CACHE = False

def find_runtime_weight(weights_dir):
    candidates = [
        weights_dir / "R50+ViT-B_16.npz",
        weights_dir / "R50-ViT-B_16.npz",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Không tìm thấy pretrained weight trong runtime: {weights_dir}")

def count_synapse_files(root):
    root = resolve_synapse_root(root)
    train_files = list((root / "train_npz").glob("*.npz"))
    test_files = list((root / "test_vol_h5").glob("*.npy.h5"))
    return root, len(train_files), len(test_files)

runtime_synapse_root, runtime_train_count, runtime_test_count = count_synapse_files(PROJECT_DIR / "data" / "Synapse")
runtime_weight_file = find_runtime_weight(PROJECT_DIR / "model" / "vit_checkpoint" / "imagenet21k")

print("Runtime dataset root:", runtime_synapse_root)
print("Runtime train slices:", runtime_train_count)
print("Runtime test volumes:", runtime_test_count)
print("Runtime weight file:", runtime_weight_file)
print("Drive dataset target:", DRIVE_DATASET_DIR)
print("Drive weight target:", DRIVE_WEIGHT_FILE)

if not PUSH_RUNTIME_CACHE_TO_DRIVE:
    print("\nSet PUSH_RUNTIME_CACHE_TO_DRIVE = True rồi chạy lại cell này nếu muốn copy dataset/weight lên MyDrive.")
else:
    if not USE_GOOGLE_DRIVE or not Path("/content/drive/MyDrive").exists():
        raise RuntimeError("Google Drive chưa sẵn sàng. Chạy cell mount/boot phía trên trước.")

    drive_dataset_target = DRIVE_DATASET_DIR
    drive_weight_target = DRIVE_WEIGHT_FILE

    if drive_dataset_target.exists():
        if not OVERWRITE_DRIVE_CACHE:
            raise FileExistsError(
                f"Drive dataset target đã tồn tại: {drive_dataset_target}. "
                "Đặt OVERWRITE_DRIVE_CACHE = True nếu muốn ghi đè."
            )
        shutil.rmtree(drive_dataset_target)

    drive_dataset_target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(runtime_synapse_root, drive_dataset_target)

    drive_weight_target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(runtime_weight_file, drive_weight_target)

    drive_dataset_root, drive_train_count, drive_test_count = count_synapse_files(drive_dataset_target)

    print("\nDrive cache completed.")
    print("Drive dataset root:", drive_dataset_root)
    print("Drive train slices:", drive_train_count)
    print("Drive test volumes:", drive_test_count)
    print("Drive weight file:", drive_weight_target)

Using Synapse dataset: /content/TransUNet-Medical-Image-Segmentation/data/Synapse
Runtime dataset root: /content/TransUNet-Medical-Image-Segmentation/data/Synapse
Runtime train slices: 2211
Runtime test volumes: 12
Runtime weight file: /content/TransUNet-Medical-Image-Segmentation/model/vit_checkpoint/imagenet21k/R50+ViT-B_16.npz
Drive dataset target: /content/drive/MyDrive/datasets/Synapse
Drive weight target: /content/drive/MyDrive/transunet/R50+ViT-B_16.npz

Set PUSH_RUNTIME_CACHE_TO_DRIVE = True rồi chạy lại cell này nếu muốn copy dataset/weight lên MyDrive.


In [6]:

import json
import os
import shlex
import subprocess
import time

import torch

from experiment_utils import build_attention_suffix, parse_attention_scales

PROFILE_TABLE = {
    "full": {"max_epochs": 150, "batch_size": 24, "base_lr": 0.01, "max_train_samples": 0},
    "colab_safe": {"max_epochs": 150, "batch_size": 2, "base_lr": 0.0008333333333333334, "max_train_samples": 0},
    "smoke": {"max_epochs": 1, "batch_size": 2, "base_lr": 0.0008, "max_train_samples": 64},
}

def gpu_memory_gb():
    if not torch.cuda.is_available():
        return 0.0
    return torch.cuda.get_device_properties(0).total_memory / 2**30

def resolve_profile():
    if RUN_PROFILE == "auto":
        return "full" if gpu_memory_gb() >= 39 else "colab_safe"
    if RUN_PROFILE not in PROFILE_TABLE:
        raise ValueError(f"Unsupported RUN_PROFILE: {RUN_PROFILE}")
    return RUN_PROFILE

def build_run_config():
    resolved_profile = resolve_profile()
    cfg = {
        "dataset": OVERRIDES["dataset"],
        "img_size": OVERRIDES["img_size"],
        "vit_name": OVERRIDES["vit_name"],
        "vit_patches_size": OVERRIDES["vit_patches_size"],
        "n_skip": OVERRIDES["n_skip"],
        "num_classes": OVERRIDES["num_classes"],
        "seed": OVERRIDES["seed"],
        "deterministic": OVERRIDES["deterministic"],
        "max_iterations": OVERRIDES["max_iterations"],
        "num_workers": OVERRIDES["num_workers"],
        "max_train_samples": OVERRIDES["max_train_samples"],
        "attention_mode": ATTENTION_MODE,
        "attention_scales_raw": ATTENTION_SCALES,
        "attention_reduction": ATTENTION_REDUCTION,
        "profile": resolved_profile,
    }
    cfg.update(PROFILE_TABLE[resolved_profile])

    for key in ("max_epochs", "batch_size", "base_lr", "max_train_samples"):
        override_value = OVERRIDES.get(key)
        if override_value not in (None, ""):
            cfg[key] = override_value

    cfg["attention_scales"] = parse_attention_scales(cfg["attention_mode"], cfg["attention_scales_raw"])
    cfg["exp"] = f"TU_{cfg['dataset']}{cfg['img_size']}"
    return cfg

def build_snapshot_name(cfg):
    name = "TU_pretrain_" + cfg["vit_name"]
    name += "_skip" + str(cfg["n_skip"])
    if cfg["vit_patches_size"] != 16:
        name += "_vitpatch" + str(cfg["vit_patches_size"])
    if cfg["max_iterations"] != 30000:
        name += "_" + str(cfg["max_iterations"])[:2] + "k"
    if cfg["max_epochs"] != 30:
        name += "_epo" + str(cfg["max_epochs"])
    name += "_bs" + str(cfg["batch_size"])
    if cfg["base_lr"] != 0.01:
        name += "_lr" + str(cfg["base_lr"])
    name += "_" + str(cfg["img_size"])
    if cfg["seed"] != 1234:
        name += "_s" + str(cfg["seed"])
    name += build_attention_suffix(cfg["attention_mode"], cfg["attention_scales"], cfg["attention_reduction"])
    if RA_MODE != "none":
        ra_scales_clean = RA_SCALES.replace(",", "")
        name += f"_ra-{RA_MODE}-s{ra_scales_clean}-r{RA_REDUCTION}"
    return name

def build_train_command(cfg):
    cmd = [
        sys.executable, "-u", "train.py",
        "--dataset", cfg["dataset"],
        "--vit_name", cfg["vit_name"],
        "--img_size", str(cfg["img_size"]),
        "--num_classes", str(cfg["num_classes"]),
        "--n_skip", str(cfg["n_skip"]),
        "--vit_patches_size", str(cfg["vit_patches_size"]),
        "--max_iterations", str(cfg["max_iterations"]),
        "--max_epochs", str(cfg["max_epochs"]),
        "--batch_size", str(cfg["batch_size"]),
        "--base_lr", str(cfg["base_lr"]),
        "--seed", str(cfg["seed"]),
        "--deterministic", str(cfg["deterministic"]),
        "--num_workers", str(cfg["num_workers"]),
        "--attention_mode", cfg["attention_mode"],
        "--attention_reduction", str(cfg["attention_reduction"]),
        "--ra_mode", RA_MODE,
        "--ra_scales", RA_SCALES,
        "--ra_reduction", str(RA_REDUCTION),
    ]
    if cfg["attention_scales"]:
        cmd += ["--attention_scales", ",".join(cfg["attention_scales"])]
    if cfg["max_train_samples"]:
        cmd += ["--max_train_samples", str(cfg["max_train_samples"])]
    return cmd

def build_test_command(cfg):
    cmd = [
        sys.executable, "-u", "test.py",
        "--dataset", cfg["dataset"],
        "--vit_name", cfg["vit_name"],
        "--img_size", str(cfg["img_size"]),
        "--num_classes", str(cfg["num_classes"]),
        "--n_skip", str(cfg["n_skip"]),
        "--vit_patches_size", str(cfg["vit_patches_size"]),
        "--max_iterations", str(cfg["max_iterations"]),
        "--max_epochs", str(cfg["max_epochs"]),
        "--batch_size", str(cfg["batch_size"]),
        "--base_lr", str(cfg["base_lr"]),
        "--seed", str(cfg["seed"]),
        "--deterministic", str(cfg["deterministic"]),
        "--attention_mode", cfg["attention_mode"],
        "--attention_reduction", str(cfg["attention_reduction"]),
        "--ra_mode", RA_MODE,
        "--ra_scales", RA_SCALES,
        "--ra_reduction", str(RA_REDUCTION),
    ]
    if cfg["attention_scales"]:
        cmd += ["--attention_scales", ",".join(cfg["attention_scales"])]
    if SAVE_NIFTI:
        cmd.append("--is_savenii")
    return cmd

def run_command(cmd, extra_env=None):
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    if extra_env:
        env.update({key: str(value) for key, value in extra_env.items()})
    existing_pythonpath = env.get("PYTHONPATH", "")
    env["PYTHONPATH"] = str(PROJECT_DIR) if not existing_pythonpath else str(PROJECT_DIR) + os.pathsep + existing_pythonpath
    printable = " ".join(shlex.quote(str(part)) for part in cmd)
    print("$", printable, flush=True)
    start = time.time()

    process = subprocess.Popen(
        [str(part) for part in cmd],
        cwd=PROJECT_DIR,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )

    captured_lines = []
    if process.stdout is not None:
        for line in process.stdout:
            line_stripped = line.rstrip('\r\n')
            if line_stripped:
                print(line_stripped, flush=True)
                captured_lines.append(line_stripped)

    return_code = process.wait()
    elapsed = (time.time() - start) / 60
    if return_code != 0:
        print(f"Command failed after {elapsed:.2f} minutes with exit code {return_code}.", flush=True)
        print("=" * 70, flush=True)
        print("LAST 50 LINES OF OUTPUT (full output above):", flush=True)
        print("=" * 70, flush=True)
        for tail_line in captured_lines[-50:]:
            print(tail_line, flush=True)
        print("=" * 70, flush=True)
        # Save full log to file for inspection
        log_path = artifact_dir / "train_failure.log"
        log_path.parent.mkdir(parents=True, exist_ok=True)
        with open(log_path, "w") as f:
            f.write("\n".join(captured_lines))
        print(f"Full log saved to: {log_path}", flush=True)
        raise subprocess.CalledProcessError(return_code, cmd, output="\n".join(captured_lines))

    print(f"Finished in {elapsed:.2f} minutes", flush=True)

run_cfg = build_run_config()
snapshot_name = build_snapshot_name(run_cfg)
snapshot_dir = PROJECT_DIR / "model" / run_cfg["exp"] / snapshot_name
resume_checkpoint_dir = DRIVE_EXPORT_DIR / "resume_checkpoints" / snapshot_name
resume_checkpoint_file = resume_checkpoint_dir / "latest_checkpoint.pth"
test_log_file = PROJECT_DIR / "test_log" / f"test_log_{run_cfg['exp']}" / f"{snapshot_name}.txt"
prediction_dir = PROJECT_DIR / "predictions" / run_cfg["exp"] / snapshot_name
artifact_dir = PROJECT_DIR / "artifacts" / snapshot_name
artifact_dir.mkdir(parents=True, exist_ok=True)
resume_checkpoint_dir.mkdir(parents=True, exist_ok=True)

print(json.dumps({**run_cfg, "attention_scales": list(run_cfg["attention_scales"])}, indent=2))
print("GPU memory (GB):", round(gpu_memory_gb(), 2))
print("Snapshot dir:", snapshot_dir)
print("Resume checkpoint dir:", resume_checkpoint_dir)
print("Resume checkpoint file exists:", resume_checkpoint_file.exists())

{
  "dataset": "Synapse",
  "img_size": 224,
  "vit_name": "R50-ViT-B_16",
  "vit_patches_size": 16,
  "n_skip": 3,
  "num_classes": 9,
  "seed": 1234,
  "deterministic": 1,
  "max_iterations": 30000,
  "num_workers": 2,
  "max_train_samples": 0,
  "attention_mode": "pre_hidden",
  "attention_scales_raw": "1/16",
  "attention_reduction": 16,
  "profile": "full",
  "max_epochs": 150,
  "batch_size": 24,
  "base_lr": 0.01,
  "attention_scales": [
    "1/16"
  ],
  "exp": "TU_Synapse224"
}
GPU memory (GB): 14.56
Snapshot dir: /content/TransUNet-Medical-Image-Segmentation/model/TU_Synapse224/TU_pretrain_R50-ViT-B_16_skip3_epo150_bs24_224_attn-pre_hidden-1_16-r16_ra-ra_skip-s0-r8
Resume checkpoint dir: /content/drive/MyDrive/transunet_colab_outputs/resume_checkpoints/TU_pretrain_R50-ViT-B_16_skip3_epo150_bs24_224_attn-pre_hidden-1_16-r16_ra-ra_skip-s0-r8
Resume checkpoint file exists: False


In [ ]:

train_env = {
    "TRANSUNET_WEIGHTS_DIR": PROJECT_DIR / "model" / "vit_checkpoint" / "imagenet21k",
    "TRANSUNET_ITER_LOG_INTERVAL": "20",
    "TRANSUNET_MID_EPOCH_SAVE_ITERS": "500",
}

if PERSIST_CHECKPOINTS_TO_DRIVE:
    train_env["TRANSUNET_CHECKPOINT_DIR"] = resume_checkpoint_dir

if RUN_TRAIN:
    run_command(build_train_command(run_cfg), extra_env=train_env)
else:
    print("RUN_TRAIN = False, skipped training.")

if snapshot_dir.exists():
    print("Checkpoint files:")
    for checkpoint_path in sorted(snapshot_dir.glob("*.pth")):
        print(" -", checkpoint_path.name)
else:
    print("Snapshot directory does not exist yet:", snapshot_dir)

print("Resume checkpoint file:", resume_checkpoint_file)

$ /usr/bin/python3 -u train.py --dataset Synapse --vit_name R50-ViT-B_16 --img_size 224 --num_classes 9 --n_skip 3 --vit_patches_size 16 --max_iterations 30000 --max_epochs 150 --batch_size 24 --base_lr 0.01 --seed 1234 --deterministic 1 --num_workers 2 --attention_mode pre_hidden --attention_reduction 16 --ra_mode ra_skip --ra_scales 0 --ra_reduction 8 --attention_scales 1/16
Namespace(root_path='./data/Synapse/train_npz', dataset='Synapse', list_dir='./splits/synapse', num_classes=9, max_iterations=30000, max_epochs=150, batch_size=24, n_gpu=1, deterministic=1, base_lr=0.01, num_workers=2, img_size=224, seed=1234, max_train_samples=0, n_skip=3, vit_name='R50-ViT-B_16', vit_patches_size=16, attention_mode='pre_hidden', attention_scales=('1/16',), attention_reduction=16, preprocess_mode='none', clahe_clip_limit=2.0, clahe_tile_grid_size=8, ra_mode='ra_skip', ra_scales='0', ra_reduction=8, is_pretrain=True, exp='TU_Synapse224', device='cuda')
The length of train set is: 2211
93 iteratio

In [ ]:

import json
from pathlib import Path

import torch

# Rebuild required runtime variables if this cell is executed after a kernel restart.
if "run_cfg" not in globals():
    run_cfg = build_run_config()

if "snapshot_name" not in globals():
    snapshot_name = build_snapshot_name(run_cfg)

if "snapshot_dir" not in globals():
    snapshot_dir = PROJECT_DIR / "model" / run_cfg["exp"] / snapshot_name

if "resume_checkpoint_dir" not in globals():
    resume_checkpoint_dir = DRIVE_EXPORT_DIR / "resume_checkpoints" / snapshot_name

if "resume_checkpoint_file" not in globals():
    resume_checkpoint_file = resume_checkpoint_dir / "latest_checkpoint.pth"

snapshot_dir.mkdir(parents=True, exist_ok=True)

def materialize_eval_checkpoints(snapshot_dir, resume_checkpoint_file, max_epochs):
    snapshot_dir = Path(snapshot_dir)
    resume_checkpoint_file = Path(resume_checkpoint_file)

    epoch_checkpoint = snapshot_dir / f"epoch_{max_epochs - 1}.pth"
    best_checkpoint = snapshot_dir / "best_model.pth"

    status = {
        "snapshot_dir": str(snapshot_dir),
        "resume_checkpoint_file": str(resume_checkpoint_file),
        "epoch_checkpoint_exists": epoch_checkpoint.exists(),
        "best_checkpoint_exists": best_checkpoint.exists(),
        "resume_exists": resume_checkpoint_file.exists(),
    }

    if epoch_checkpoint.exists() or best_checkpoint.exists():
        print(json.dumps(status, indent=2))
        return epoch_checkpoint, best_checkpoint

    if not resume_checkpoint_file.exists():
        raise FileNotFoundError(
            f"Resume checkpoint not found: {resume_checkpoint_file}. "
            "Run the training cell again or verify the Drive checkpoint directory."
        )

    resume_state = torch.load(resume_checkpoint_file, map_location="cpu")
    state_dict = resume_state["model_state"] if isinstance(resume_state, dict) and "model_state" in resume_state else resume_state

    torch.save(state_dict, epoch_checkpoint)
    torch.save(state_dict, best_checkpoint)

    status["epoch_checkpoint_exists"] = epoch_checkpoint.exists()
    status["best_checkpoint_exists"] = best_checkpoint.exists()
    print(json.dumps(status, indent=2))
    print("Materialized local evaluation checkpoints:")
    print(" -", epoch_checkpoint)
    print(" -", best_checkpoint)
    return epoch_checkpoint, best_checkpoint

materialize_eval_checkpoints(snapshot_dir, resume_checkpoint_file, run_cfg["max_epochs"])

In [ ]:

from pathlib import Path

def ensure_test_cli_accepts_snapshot_args(test_py_path):
    test_py_path = Path(test_py_path)
    text = test_py_path.read_text(encoding="utf-8")
    if "--max_iterations" in text and "--max_epochs" in text:
        return
    anchor = "parser.add_argument('--vit_patches_size', type=int, default=16, help='vit_patches_size, default is 16')\n"
    if anchor not in text:
        raise RuntimeError(f"Cannot patch evaluation CLI; anchor not found in {test_py_path}")
    insert = (
        anchor
        + "parser.add_argument('--max_iterations', type=int, default=30000,\n"
        + "                    help='max iterations used to reconstruct the training snapshot name during evaluation')\n"
        + "parser.add_argument('--max_epochs', type=int, default=30,\n"
        + "                    help='max epochs used to reconstruct the training snapshot name during evaluation')\n"
    )
    test_py_path.write_text(text.replace(anchor, insert, 1), encoding="utf-8")
    print(f"Patched evaluation CLI args in: {test_py_path}")

ensure_test_cli_accepts_snapshot_args(PROJECT_DIR / "test.py")

test_env = {}

if PERSIST_CHECKPOINTS_TO_DRIVE:
    test_env["TRANSUNET_CHECKPOINT_DIR"] = str(resume_checkpoint_dir)

if RUN_TEST:
    if not snapshot_dir.exists() and not resume_checkpoint_file.exists():
        raise FileNotFoundError(
            f"Neither local snapshot dir nor resume checkpoint exists. Checked {snapshot_dir} and {resume_checkpoint_file}."
        )
    run_command(build_test_command(run_cfg), extra_env=test_env)
else:
    print("RUN_TEST = False, skipped evaluation.")

print("Expected test log:", test_log_file)
print("Prediction directory:", prediction_dir)


In [ ]:

import json
import re
import zipfile

def parse_metrics_from_log(log_file):
    if not log_file.exists():
        return {
            "overall": {"mean_dice": None, "mean_hd95": None},
            "per_class": [],
            "log_found": False,
        }

    text = log_file.read_text(encoding="utf-8")
    overall_match = re.search(
        r"Testing performance in best val model: mean_dice : ([0-9.]+) mean_hd95 : ([0-9.]+)",
        text,
    )
    class_matches = re.findall(
        r"Mean class (\d+) mean_dice ([0-9.]+) mean_hd95 ([0-9.]+)",
        text,
    )
    return {
        "overall": {
            "mean_dice": float(overall_match.group(1)) if overall_match else None,
            "mean_hd95": float(overall_match.group(2)) if overall_match else None,
        },
        "per_class": [
            {"class_id": int(cid), "mean_dice": float(dice), "mean_hd95": float(hd95)}
            for cid, dice, hd95 in class_matches
        ],
        "log_found": True,
    }

def add_path_to_zip(zip_file, source, arcname):
    source = Path(source)
    if not source.exists():
        return
    if source.is_file():
        zip_file.write(source, arcname)
        return
    for file_path in sorted(source.rglob("*")):
        if file_path.is_file():
            zip_file.write(file_path, Path(arcname) / file_path.relative_to(source))

metrics = parse_metrics_from_log(test_log_file)
summary = {
    "project_dir": str(PROJECT_DIR),
    "config": {**run_cfg, "attention_scales": list(run_cfg["attention_scales"])},
    "paths": {
        "snapshot_dir": str(snapshot_dir),
        "test_log_file": str(test_log_file),
        "prediction_dir": str(prediction_dir),
        "artifact_dir": str(artifact_dir),
    },
    "metrics": metrics,
}

metrics_path = artifact_dir / "metrics.json"
metrics_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(json.dumps(summary["metrics"], indent=2))
print("Metrics file:", metrics_path)

zip_path = artifact_dir / f"{snapshot_name}.zip"
if ZIP_ARTIFACTS:
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zip_file:
        add_path_to_zip(zip_file, snapshot_dir, "model")
        add_path_to_zip(zip_file, test_log_file, f"logs/{test_log_file.name}")
        add_path_to_zip(zip_file, metrics_path, "metrics.json")
        if prediction_dir.exists():
            add_path_to_zip(zip_file, prediction_dir, "predictions")
    print("Artifact zip:", zip_path)
else:
    print("ZIP_ARTIFACTS = False")

if EXPORT_TO_DRIVE and USE_GOOGLE_DRIVE and IN_COLAB:
    DRIVE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy2(metrics_path, DRIVE_EXPORT_DIR / metrics_path.name)
    if ZIP_ARTIFACTS and zip_path.exists():
        shutil.copy2(zip_path, DRIVE_EXPORT_DIR / zip_path.name)
    print("Copied exports to:", DRIVE_EXPORT_DIR)
else:
    print("Drive export skipped.")

## Sau khi notebook chạy xong

1. Tải hoặc mở file `metrics.json` được in ở cell export cuối.
2. Ghi lại các trường quan trọng:
   - `metrics.overall.mean_dice`
   - `metrics.overall.mean_hd95`
   - class 6 trong `metrics.per_class` là pancreas.
4. Nếu `metrics.overall.mean_dice` hoặc `mean_hd95` là `null`, kiểm tra `test_log_file` vì bước evaluation chưa hoàn tất hoặc log chưa được ghi.
